In [1]:
# Initial Python environment setup...
import numpy as np # linear algebra
import pandas as pd # CSV file I/O (e.g. pd.read_csv)
import os # reading the input files we have access to
import time
import sys

# Multiprocessing
# from multiprocessing import Pool

# Import Sklearn and lgbm
import lightgbm as lgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

#Visualization

print(os.listdir('../input'))


['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
# Test Unit
# training_step = list(range(0, 100000, 10000))
# model = None
# origin_cols = ['key',
#  'fare_amount',
#  'pickup_datetime',
#  'pickup_longitude',
#  'pickup_latitude',
#  'dropoff_longitude',
#  'dropoff_latitude',
#  'passenger_count']

# for i in range(1, len(training_step)):
#     start = time.time()
#     data = pd.read_csv("../input/train.csv", skiprows=training_step[i-1], nrows=training_step[i])
#     data.columns = origin_cols
#     model = incremental_learning(data, model)
#     print("TRAINING NUMBER:", i,"Elapsed:", time.time() - start, "s\n")
    

# predict_and_submit(model, submit=False)


In [3]:
# Main Pipeline
def incremental_learning(data, model):
    start = time.time()
    data["pickup_datetime"] = pd.to_datetime(data["pickup_datetime"], infer_datetime_format=True)
    print("Conversion took", time.time() - start, "s")
    add_travel_vector_features(data)
    data = pd.concat([data, timeExpansion(data["pickup_datetime"])], axis=1)

    X_tr, X_te, y_tr, y_te = train_test_split(
        data.drop(["key", "pickup_datetime", "fare_amount"], axis=1), 
        data['fare_amount'], 
        test_size=0.2)
    
    params = {
        'boosting_type': 'gbdt',
        'objective': 'regression_l1',
        'metric': {'l1'},
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': 0
    }

    
    dset = lgbm.Dataset(X_tr, y_tr, free_raw_data=True)
    model = lgbm.train(params, 
                       init_model = model,
                       train_set = dset,
                       keep_training_booster = True,
                       num_boost_round = 5)
    MAE = mean_absolute_error(y_te, model.predict(X_te))
    print("Mean Absolute Error:", MAE)
    return model


In [4]:
# Helper Function

# def PandasMultiprocessing(Series, function, pool):
#     start = time.time()
#     n_cores = 4
#     lin = np.linspace(0, len(Series), n_cores+1, dtype="int")
    
#     print("Start Multiprocessing")
    
#     result = pool.starmap_async(function, [[Series[lin[num-1]:lin[num]]] for num in range(1, len(lin))])    
    
#     converted = pd.concat(result.get())
    
#     print("Multiprocessing Took", time.time() - start, "s")
#     return converted

def partial_import(filename, skip, rows):
    return pd.read_csv('..input/train.csv', skiprows=skip, nrows=rows)

def add_travel_vector_features(df):
    df['abs_diff_longitude'] = (df.dropoff_longitude - df.pickup_longitude).abs()
    df['abs_diff_latitude'] = (df.dropoff_latitude - df.pickup_latitude).abs()
    
def month_translation(number):
    month_list = ["jan", "feb", "mar", 
                   "apr", "may", "jun", "jul", 
                   "aug", "sep", "oct", "nov", "dec"]
    
    for num, i in enumerate(month_list):
        if num == number-1:
            return i

def quarter_translation(number):
    quarter_border = [0, 3, 6, 9, 12, 15, 18, 21, 24]
    quarter_list = ["11", "12", "13", "14", "21", "22", "23", "24"]
    for num, i in enumerate(range(1, len(quarter_border))):
        if (number < quarter_border[i]) & (number >= quarter_border[i-1]):
            return quarter_list[num]

def week_translation(number):
    week_border = [0, 7, 14, 21, 31]
    week_list = ["1W", "2W", "3W", "4W"]
    for num, i in enumerate(range(1, len(week_border))):
         if (number <= week_border[i]) & (number > week_border[i-1]):
            return week_list[num]


# add year, 12 month, week, and 8 quarter time status
def timeExpansion(timeSeries):
    additional_cols = ["year", "month", "quarter", "week"]
    dummy0 = pd.DataFrame(np.zeros((len(timeSeries), len(additional_cols)), 'int'), 
                         columns=additional_cols, 
                         index=timeSeries.index)
    
    dummy0["year"] = [i.year for i in timeSeries]
    dummy0["month"] = [month_translation(i.month) for i in timeSeries]
    dummy0["day"] = [week_translation(i.day) for i in timeSeries]
    dummy0["quarter"] = [quarter_translation(i.hour) for i in timeSeries]
    return pd.get_dummies(dummy0)

def predict_and_submit(estimator, submit=True):
    test_df = pd.read_csv('../input/test.csv')
    add_travel_vector_features(test_df)
    test_df["pickup_datetime"] = pd.to_datetime(test_df["pickup_datetime"])
    test_X = pd.concat([test_df, timeExpansion(test_df["pickup_datetime"])], axis=1)
    test_X.drop(["key", "pickup_datetime"], axis=1, inplace=True)

    print("Test Shape: ",test_X.shape)

    # Predict fare_amount on the test set using our model (w) trained on the training set.
    test_y_predictions = estimator.predict(test_X)

    # Write the predictions to a CSV file which we can submit to the competition.
    if submit:
        submission = pd.DataFrame(
            {'key': test_df.key, 'fare_amount': test_y_predictions},
            columns = ['key', 'fare_amount'])
        submission.to_csv('submission.csv', index = False)


In [5]:
%%time

#MAIN Execution
training_step = list(range(0, 15000000, 5000000))
estimator = None
origin_cols = ['key',
 'fare_amount',
 'pickup_datetime',
 'pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude',
 'passenger_count']

for i in range(1, len(training_step)):
    start = time.time()
    print(training_step[i-1], training_step[i])
    train_df = pd.read_csv("../input/train.csv", skiprows=training_step[i-1], nrows=training_step[i])
    train_df.columns = origin_cols
    estimator = incremental_learning(train_df, estimator)
    print("Operation", i, "took ", time.time() - start, " \n")

predict_and_submit(estimator)


0 5000000


/tmp/ipykernel_11/1148623683.py:4: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data["pickup_datetime"] = pd.to_datetime(data["pickup_datetime"], infer_datetime_format=True)


Conversion took 30.697513580322266 s
Mean Absolute Error: 4.076279944739375
Operation 1 took  81.62561798095703  

5000000 10000000


/tmp/ipykernel_11/1148623683.py:4: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data["pickup_datetime"] = pd.to_datetime(data["pickup_datetime"], infer_datetime_format=True)


Conversion took 61.94406747817993 s
Mean Absolute Error: 3.194483798766101
Operation 2 took  188.01412439346313  

Test Shape:  (9914, 33)
CPU times: user 4min 39s, sys: 7.39 s, total: 4min 47s
Wall time: 4min 29s
